# NeuralTuner TRL Training Notebook (Mac-first)

This notebook is for training and evaluation only. Inference-only logic lives in `inference.py`.

Model default: `Qwen/Qwen2.5-1.5B-Instruct`


In [2]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from server.neural_tuner_env_environment import NeuralTunerEnvironment
from models import NeuralTunerAction


In [3]:
# Optional: set token in shell before launching Jupyter, or via .env.local
# Example in terminal: export HF_TOKEN=your_token_here
import os
# os.environ["HF_TOKEN"] = ""
print("HF_TOKEN loaded:", bool(os.getenv("HF_TOKEN")))

HF_TOKEN loaded: False


## 1) Environment smoke test
Validate reset/step/state before training.

In [4]:
env = NeuralTunerEnvironment()
reset_obs = env.reset(model_id="inception_v3", difficulty="medium", seed=42)
print(reset_obs.output[:800])

bench = env.step(NeuralTunerAction(action_type="benchmark"))
print(bench.output)
print(env.state)


NEURAL TUNER  —  INCEPTION V3
Model:       Inception V3  (47.0M params)
Description: Google Inception V3 image classifier (47M params).
Difficulty:  MEDIUM

BASELINE
  Latency: 175.1 ms
  Memory:  186.1 MB

SNAPDRAGON HTP CONSTRAINTS
  Latency budget:  78.8 ms
  Memory budget:   83.7 MB
  Min accuracy:    0.9

SCENARIO
  Tight latency target for a flagship Snapdragon mobile deployment. Must use INT4 selectively while keeping classifier accuracy intact.

LAYERS (10 total)
  Layer ID                     Type                       Latency    Memory  Sensitivity
  -------------------------------------------------------------------------------------
  conv_stem      
BENCHMARK 1/5
  Latency:   175.10 ms  (budget 78.8 ms)  [FAIL]  ↓0.0%
  Memory:    186.10 MB  (budget 83.7 MB)  [FAIL]
  Accuracy: 1.0000  (min 0.9)  [PASS]
  Projected reward: 0.2000
  All constraints met: NO
  Benchmarks remaining: 4
episode_id='51ed2c92-23f2-46fa-9fc9-f4b167d2021f' step_count=1 model_id='inception_v3' diffic

## 2) Baseline data collection
Collect random/untrained policy episodes for comparison.

In [5]:
from rollout_eval import run_baseline_episode, run_heuristic_episode

env = NeuralTunerEnvironment()
baseline = run_baseline_episode(env, "inception_v3", "medium")
heuristic = run_heuristic_episode(env, "inception_v3", "medium")
print(baseline)
print(heuristic)


EpisodeMetrics(policy='baseline', episode_index=0, model_id='inception_v3', difficulty='medium', final_reward=0.786, done=True, step_count=10, benchmark_count=1, latency_ms=64.18, memory_mb=40.01, accuracy_retention=0.9663, constraints_met=True)
EpisodeMetrics(policy='heuristic', episode_index=0, model_id='inception_v3', difficulty='medium', final_reward=0.6428, done=True, step_count=14, benchmark_count=1, latency_ms=93.77, memory_mb=77.11, accuracy_retention=0.9785, constraints_met=False)


## 3) Qwen + TRL OpenEnv setup (tiny run)
This section follows the TRL OpenEnv notebook pattern (`environment_factory`) but uses the local NeuralTuner environment and low-cost defaults for first-run debugging.

In [ ]:
from huggingface_hub import snapshot_download

HF_TOKEN = os.getenv("HF_TOKEN")
LOCAL_MODEL_DIR = Path(".hf_cache/Qwen--Qwen2.5-1.5B-Instruct").resolve()
LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id="Qwen/Qwen2.5-1.5B-Instruct",
    local_dir=str(LOCAL_MODEL_DIR),
    token=HF_TOKEN,
)

MODEL_NAME = str(LOCAL_MODEL_DIR)  

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
from dataclasses import asdict
from typing import Optional
from pathlib import Path
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer


MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "outputs/neural_tuner_grpo"
TRAIN_EPISODES = 24
NUM_EPOCHS = 1
SYSTEM_PROMPT = """You are a hardware optimization agent for NeuralTuner.
Use available tools to profile layers, apply quantization, benchmark, and submit.
Prioritize meeting latency, memory, and accuracy constraints while maximizing reward.
"""


class NeuralTunerOpenEnv:
    """OpenEnv wrapper compatible with TRL environment_factory."""

    scenario_schedule: list[dict] = []
    schedule_idx: int = 0

    def __init__(self):
        self._env = NeuralTunerEnvironment()
        self.reward = 0.0
        self.done = False

    def reset(self, **kwargs) -> str:
        scenario = None
        if kwargs.get("model_id") or kwargs.get("difficulty"):
            scenario = {
                "model_id": kwargs.get("model_id", "inception_v3"),
                "difficulty": kwargs.get("difficulty", "medium"),
            }
        elif self.scenario_schedule:
            scenario = self.scenario_schedule[self.schedule_idx % len(self.scenario_schedule)]
            NeuralTunerOpenEnv.schedule_idx += 1
        else:
            scenario = {"model_id": "inception_v3", "difficulty": "medium"}

        obs = self._env.reset(
            difficulty=scenario["difficulty"],
            model_id=scenario["model_id"],
            seed=kwargs.get("seed", 42),
        )
        self.reward = 0.0
        self.done = False
        return obs.output

    def _step(self, action_type: str, layer_id: Optional[str] = None, dtype: Optional[str] = None) -> str:
        result = self._env.step(
            NeuralTunerAction(action_type=action_type, layer_id=layer_id, dtype=dtype)
        )
        self.reward = float(result.reward)
        self.done = bool(result.done)
        return result.output

    def profile_layer(self, layer_id: str) -> str:
        """Reveal layer sensitivity and quantization risk for one layer."""
        return self._step("profile_layer", layer_id=layer_id)

    def quantize_layer(self, layer_id: str, dtype: str) -> str:
        """Apply dtype quantization to one layer. dtype in FP32/FP16/INT8/INT4."""
        return self._step("quantize_layer", layer_id=layer_id, dtype=dtype)

    def revert_layer(self, layer_id: str) -> str:
        """Revert a quantized layer back to FP32."""
        return self._step("revert_layer", layer_id=layer_id)

    def benchmark(self) -> str:
        """Run simulated hardware benchmark for current quantization map."""
        return self._step("benchmark")

    def submit(self) -> str:
        """Submit final configuration and finish the episode."""
        return self._step("submit")


def reward_func(environments, **kwargs) -> list[float]:
    return [float(env.reward) for env in environments]


MOCK_DATA_PATH = Path("data/mock_data/neural_tuner_train_scenarios.jsonl")


def _load_mock_scenarios(path: Path) -> list[dict]:
    if not path.exists():
        raise FileNotFoundError(f"Mock training data not found: {path}")
    rows = []
    with path.open() as fp:
        for line in fp:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    if not rows:
        raise ValueError(f"No scenarios found in {path}")
    return rows


def _build_prompt_row(scenario: dict) -> list[dict]:
    scenario_text = (
        f"Scenario ID: {scenario['id']}\n"
        f"Model: {scenario['model_id']}\n"
        f"Difficulty: {scenario['difficulty']}\n"
        f"Hint: {scenario['scenario_hint']}\n"
        f"Target behavior: {scenario['target_behavior']}\n"
        f"Notes: {scenario['notes']}\n"
    )
    return [{"role": "user", "content": SYSTEM_PROMPT + "\n\n" + scenario_text}]


mock_scenarios = _load_mock_scenarios(MOCK_DATA_PATH)
expanded = (mock_scenarios * ((TRAIN_EPISODES + len(mock_scenarios) - 1) // len(mock_scenarios)))[:TRAIN_EPISODES]

# Align environment resets with mock training prompts.
NeuralTunerOpenEnv.scenario_schedule = expanded
NeuralTunerOpenEnv.schedule_idx = 0

train_dataset = Dataset.from_dict(
    {
        "prompt": [_build_prompt_row(s) for s in expanded],
        "model_id": [s["model_id"] for s in expanded],
        "difficulty": [s["difficulty"] for s in expanded],
        "scenario_id": [s["id"] for s in expanded],
    }
)
print(f"dataset size: {len(train_dataset)} episodes")
print(f"model: {MODEL_NAME}")
print(f"mock source: {MOCK_DATA_PATH}")
print("sample prompt:")
print(train_dataset[0]["prompt"][0]["content"][:450])


dataset size: 24 episodes
model: Qwen/Qwen2.5-1.5B-Instruct
mock source: data/mock_data/neural_tuner_train_scenarios.jsonl
sample prompt:
You are a hardware optimization agent for NeuralTuner.
Use available tools to profile layers, apply quantization, benchmark, and submit.
Prioritize meeting latency, memory, and accuracy constraints while maximizing reward.


Scenario ID: s1
Model: inception_v3
Difficulty: easy
Hint: Start by profiling the largest inception blocks and avoid quantizing classifier aggressively.
Target behavior: profile_first_then_quantize
Notes: Prefer INT8 on low s


## 4) Save baseline + tiny training dataset
This saves baseline metrics and prompt data so you can trace what was used for training.

In [ ]:
from rollout_eval import run_baseline_episode, run_heuristic_episode

artifacts_dir = Path("artifacts/training")
artifacts_dir.mkdir(parents=True, exist_ok=True)

# Baseline before training
baseline_env = NeuralTunerEnvironment()
baseline_metrics = run_baseline_episode(baseline_env, "inception_v3", "medium")
heuristic_metrics = run_heuristic_episode(baseline_env, "inception_v3", "medium")

baseline_path = artifacts_dir / "baseline_metrics.jsonl"
with baseline_path.open("w") as fp:
    fp.write(json.dumps(asdict(baseline_metrics)) + "\n")
    fp.write(json.dumps(asdict(heuristic_metrics)) + "\n")

# Persist tiny training prompts
prompts_path = artifacts_dir / "train_prompts.jsonl"
with prompts_path.open("w") as fp:
    for row in train_dataset:
        fp.write(json.dumps(row) + "\n")

print("Baseline metrics:")
print(asdict(baseline_metrics))
print(asdict(heuristic_metrics))
print(f"Saved baseline file: {baseline_path}")
print(f"Saved prompts file: {prompts_path}")


Baseline metrics:
{'policy': 'baseline', 'episode_index': 0, 'model_id': 'inception_v3', 'difficulty': 'medium', 'final_reward': 0.786, 'done': True, 'step_count': 10, 'benchmark_count': 1, 'latency_ms': 64.18, 'memory_mb': 40.01, 'accuracy_retention': 0.9663, 'constraints_met': True}
{'policy': 'heuristic', 'episode_index': 0, 'model_id': 'inception_v3', 'difficulty': 'medium', 'final_reward': 0.6428, 'done': True, 'step_count': 14, 'benchmark_count': 1, 'latency_ms': 93.77, 'memory_mb': 77.11, 'accuracy_retention': 0.9785, 'constraints_met': False}
Saved baseline file: artifacts/training/baseline_metrics.jsonl
Saved prompts file: artifacts/training/train_prompts.jsonl


## 5) Train tiny GRPO run, save checkpoint, print training results

In [ ]:
grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=1e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    warmup_steps=2,
    max_completion_length=384,
    num_generations=2,
    generation_batch_size=2,
    logging_steps=1,
    save_steps=20,
    save_total_limit=1,
    report_to="none",
    use_vllm=False,
)

trainer = GRPOTrainer(
    model=MODEL_NAME,
    reward_funcs=reward_func,
    train_dataset=train_dataset,
    args=grpo_config,
    environment_factory=NeuralTunerOpenEnv,
)

trainer_stats = trainer.train()
trainer.save_model(OUTPUT_DIR)

metrics_path = Path("artifacts/training/train_metrics.json")
metrics_path.parent.mkdir(parents=True, exist_ok=True)
metrics_path.write_text(json.dumps(trainer_stats.metrics, indent=2))

print("Training complete.")
print(f"Saved checkpoint: {OUTPUT_DIR}")
print(f"Saved metrics: {metrics_path}")
print("Key metrics:")
for key in ["train_runtime", "train_samples_per_second", "train_steps_per_second", "train_loss"]:
    if key in trainer_stats.metrics:
        print(f"  {key}: {trainer_stats.metrics[key]}")

# Quick post-training comparison hook (environment-side, deterministic reference)
post_env = NeuralTunerEnvironment()
post_baseline = run_baseline_episode(post_env, "inception_v3", "medium")
post_heuristic = run_heuristic_episode(post_env, "inception_v3", "medium")

comparison_df = pd.DataFrame([
    {"stage": "pre", "policy": "baseline", "reward": baseline_metrics.final_reward},
    {"stage": "pre", "policy": "heuristic", "reward": heuristic_metrics.final_reward},
    {"stage": "post", "policy": "baseline", "reward": post_baseline.final_reward},
    {"stage": "post", "policy": "heuristic", "reward": post_heuristic.final_reward},
])

print("\nPre/Post reference rewards:")
print(comparison_df)

plot_path = Path("artifacts/plots/pre_post_reference_rewards.png")
plot_path.parent.mkdir(parents=True, exist_ok=True)
for policy, grp in comparison_df.groupby("policy"):
    plt.plot(grp["stage"], grp["reward"], marker="o", label=policy)
plt.xlabel("Stage")
plt.ylabel("Reward")
plt.title("NeuralTuner reference rewards (pre vs post)")
plt.legend()
plt.tight_layout()
plt.savefig(plot_path)
print(f"Saved plot: {plot_path}")


0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 